# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

This is the **single working notebook** for Paper 1. GitHub is the source of truth.

> **Current stage:** Sprint 1 — corpus audit, corpus reconciliation and temporal reconstruction.  
> We do **not** build semantic networks until the temporal backbone is defensible.


## Colab ↔ GitHub workflow

1. Open this notebook directly from `ardominguezm/golden-age-semantic-reconfiguration`.
2. Run **Runtime → Run all**.
3. Inspect the scientific checkpoints.
4. Preserve the run with **File → Save a copy in GitHub**, overwriting this same file on `main`.
5. Do not create parallel experimental notebooks.

Each code update may temporarily clear outputs; the previous executed state remains preserved in Git history.


## 00. Environment & reproducibility

The two upstream corpora are public and are pinned to exact commits. This means future changes in those repositories cannot silently change our results.


In [ ]:
import sys, os, re, shutil, subprocess, unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")


## 01. Authoritative sources

We audit two complementary sources:

- **Hernández-Lorenzo network corpus**: the exact derivative corpus used in the previous network study, with standardized orthography and a metadata table.
- **CorpusSonetosSigloDeOro (Navarro Colorado)**: individual TEI/XML sonnets with titles, textual structure and bibliographic source information.

The key design question is whether the Hernández aggregate TXT files can be reliably split back into poems and reconciled with the TEI corpus.


In [ ]:
SOURCES = {
    "hernandez_network": {
        "repo": "https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git",
        "commit": "ef6b7b691f67abe60d9cfa85c274f0be8095dd9a",
    },
    "navarro_tei": {
        "repo": "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "commit": "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    },
}

SOURCE_ROOT = Path("/content/gasr_sources")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

def clone_at_commit(name, repo_url, commit):
    target = SOURCE_ROOT / name
    if target.exists():
        shutil.rmtree(target)
    subprocess.run(["git","clone","--quiet",repo_url,str(target)], check=True)
    subprocess.run(["git","-C",str(target),"checkout","--quiet",commit], check=True)
    resolved = subprocess.check_output(
        ["git","-C",str(target),"rev-parse","HEAD"], text=True
    ).strip()
    assert resolved == commit, (name, resolved, commit)
    return target

source_paths = {
    name: clone_at_commit(name, info["repo"], info["commit"])
    for name, info in SOURCES.items()
}

print("Pinned sources ready:")
for name, path in source_paths.items():
    print(f"  {name}: {path}")


## 02. Hernández-Lorenzo corpus: what is the temporal metadata actually measuring?

The previous corpus contains `corpus/metadata.csv`. Before using its `Date` field we test what it represents.

A range such as `1547–1616` for Cervantes is an **author lifespan**, not a poem date. Treating it as poem chronology would reproduce the main temporal limitation we want to overcome.


In [ ]:
hernandez_root = source_paths["hernandez_network"]
h_corpus = hernandez_root / "corpus"
metadata_path = h_corpus / "metadata.csv"

h_meta = pd.read_csv(metadata_path)
h_meta = h_meta.loc[:, ~h_meta.columns.astype(str).str.startswith("Unnamed")]
print(f"Metadata rows: {len(h_meta)}")
display(h_meta.head(10))

life_pat = re.compile(r"^\s*(\d{4})\s*-\s*(\d{4})\s*$")
life = h_meta["Date"].astype(str).str.extract(life_pat)
h_meta["birth_year"] = pd.to_numeric(life[0], errors="coerce")
h_meta["death_year"] = pd.to_numeric(life[1], errors="coerce")

n_life = h_meta["birth_year"].notna().sum()
print(f"\nRows whose Date field is a YYYY-YYYY lifespan: {n_life}/{len(h_meta)}")
print("Conclusion: Hernández 'Date' is biographical metadata, not poem-level dating.")


## 03. Recover poem boundaries from the Hernández aggregate TXT files

The first run showed blank lines between consecutive sonnets. We now test this systematically across all author files.

If this works, the Hernández corpus can become our **standardized textual backbone**, while Navarro TEI contributes poem-level titles and bibliographic information for overlapping material.


In [ ]:
txt_files = sorted(h_corpus.glob("*.txt"))

def split_txt_poems(path: Path):
    raw = path.read_text(encoding="utf-8", errors="replace")
    raw = raw.replace("\r\n", "\n").replace("\r", "\n").strip()
    blocks = re.split(r"\n[ \t]*\n+", raw)
    records = []
    for j, block in enumerate(blocks, 1):
        lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
        if not lines:
            continue
        records.append({
            "source_file": path.name,
            "author_file": re.sub(r"_Sonetos.*$", "", path.stem, flags=re.I),
            "poem_local_id": j,
            "n_lines": len(lines),
            "text": "\n".join(lines),
        })
    return records

h_records = []
for p in txt_files:
    h_records.extend(split_txt_poems(p))

h_poems = pd.DataFrame(h_records)

print(f"TXT author files: {len(txt_files)}")
print(f"Recovered poem blocks: {len(h_poems):,}")
print(f"14-line blocks: {(h_poems.n_lines == 14).sum():,} / {len(h_poems):,}")
print("\nLine-count distribution:")
print(h_poems["n_lines"].value_counts().sort_index().to_string())

h_author_counts = (
    h_poems.groupby(["source_file","author_file"])
    .agg(recovered_poems=("poem_local_id","count"),
         blocks_14_lines=("n_lines", lambda s: int((s==14).sum())))
    .reset_index()
    .sort_values("recovered_poems", ascending=False)
)
display(h_author_counts)


### 03.1 Sanity checks against known author totals

We do not force a filename-to-author mapping yet. Instead, we verify several unambiguous cases where the metadata total is known. This is a quick test of whether blank-line splitting is trustworthy.


In [ ]:
def norm_letters(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r"[^a-z]", "", s.lower())

checks = {
    "Cervantes": ("Cervantes, Miguel de",),
    "Garcilaso": ("Vega, Garcilaso de la",),
    "Gongora": ("Góngora, Luis de",),
    "Quevedo": ("Quevedo, Francisco de",),
}

rows = []
for file_key, author_candidates in checks.items():
    fmask = h_author_counts["author_file"].map(norm_letters).str.contains(norm_letters(file_key), regex=False)
    file_rows = h_author_counts[fmask]
    meta_rows = h_meta[h_meta["Author"].isin(author_candidates)]
    rows.append({
        "file_key": file_key,
        "recovered": int(file_rows["recovered_poems"].iloc[0]) if len(file_rows) else None,
        "metadata_expected": int(meta_rows["Poems"].iloc[0]) if len(meta_rows) else None,
        "match": (
            int(file_rows["recovered_poems"].iloc[0]) == int(meta_rows["Poems"].iloc[0])
            if len(file_rows) and len(meta_rows) else None
        ),
    })
display(pd.DataFrame(rows))


## 04. Navarro TEI: poem-level reconstruction

The first run successfully parsed all 5,078 XML files. We repeat the parsing in a compact form and preserve only fields relevant to Paper 1.

Importantly, **bibliographic edition dates are not automatically poem composition dates**.


In [ ]:
navarro_root = source_paths["navarro_tei"]
xml_files = sorted(p for p in navarro_root.rglob("*.xml") if ".git" not in p.parts)
NS = {"tei": "http://www.tei-c.org/ns/1.0"}

def text_content(el):
    if el is None:
        return None
    return " ".join(" ".join(el.itertext()).split())

def parse_tei_poem(path, root):
    tree = ET.parse(path)
    tei = tree.getroot()
    title_el = tei.find(".//tei:text/tei:body/tei:head/tei:title", NS)
    author_el = tei.find(".//tei:sourceDesc//tei:author", NS)
    bibl_el = tei.find(".//tei:sourceDesc//tei:bibl", NS)
    line_els = tei.findall(".//tei:text/tei:body//tei:l", NS)
    lines = [text_content(x) for x in line_els]
    lines = [x for x in lines if x]
    rel = path.relative_to(root)
    return {
        "poem_id": str(rel.with_suffix("")),
        "author_dir": rel.parts[0],
        "author_tei": text_content(author_el),
        "title": text_content(title_el),
        "n_lines": len(lines),
        "text": "\n".join(lines),
        "source_bibl": text_content(bibl_el),
        "source_file": str(rel),
    }

records, parse_errors = [], []
for p in xml_files:
    try:
        records.append(parse_tei_poem(p, navarro_root))
    except Exception as exc:
        parse_errors.append((str(p.relative_to(navarro_root)), repr(exc)))

n_poems = pd.DataFrame(records)

print(f"Parsed poems: {len(n_poems):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Authors/folders: {n_poems['author_dir'].nunique():,}")
print(f"14-line sonnets: {(n_poems['n_lines']==14).sum():,} / {len(n_poems):,}")
display(n_poems[["poem_id","author_dir","author_tei","title","n_lines","source_bibl"]].head(10))


## 05. Correct temporal audit of the TEI

### Why this cell replaces the previous temporal check

The first notebook used the regex:

`(date|when|notBefore|notAfter|from|to)`

against raw XML text. The substring `to` occurs in ordinary words and tags such as *cuarteto*, *terceto* and many other places, so the reported **5,078 “date-like files” was a false positive**.

We now inspect the XML structurally:

- exact `<date>` elements;
- exact attributes `when`, `notBefore`, `notAfter`, `from`, `to`;
- and the XML ancestry of each date so we can distinguish a **witness/edition date** from a genuine poem date.


In [ ]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1] if "}" in tag else tag

TEMP_ATTRS = {"when","notBefore","notAfter","from","to"}
date_records = []

for p in xml_files:
    tree = ET.parse(p)
    root = tree.getroot()
    parent = {child: par for par in root.iter() for child in par}

    for el in root.iter():
        tag = local_name(el.tag)

        if tag == "date":
            value = text_content(el)
            ancestors = []
            cur = parent.get(el)
            for _ in range(5):
                if cur is None:
                    break
                ancestors.append(local_name(cur.tag))
                cur = parent.get(cur)
            date_records.append({
                "file": str(p.relative_to(navarro_root)),
                "kind": "date_element",
                "field": "date",
                "value": value,
                "element": tag,
                "context": ">".join(ancestors),
            })

        for attr, value in el.attrib.items():
            attr_local = local_name(attr)
            if attr_local in TEMP_ATTRS:
                ancestors = []
                cur = el
                for _ in range(5):
                    if cur is None:
                        break
                    ancestors.append(local_name(cur.tag))
                    cur = parent.get(cur)
                date_records.append({
                    "file": str(p.relative_to(navarro_root)),
                    "kind": "date_attribute",
                    "field": attr_local,
                    "value": value,
                    "element": tag,
                    "context": ">".join(ancestors),
                })

tei_dates = pd.DataFrame(date_records)

print(f"Exact TEI date/temporal records: {len(tei_dates):,}")
print(f"Files with at least one exact temporal record: {tei_dates['file'].nunique() if len(tei_dates) else 0:,}")

if len(tei_dates):
    context_summary = (
        tei_dates.groupby(["kind","field","element","context"], dropna=False)
        .agg(records=("file","size"),
             files=("file","nunique"),
             example_value=("value","first"),
             example_file=("file","first"))
        .reset_index()
        .sort_values(["files","records"], ascending=False)
    )
    display(context_summary.head(30))
    display(tei_dates.head(30))
else:
    print("No exact TEI temporal fields were found.")


### 05.1 Explicitly classify witness/edition dates

A date inside a path such as

`date > witness > listWit > sourceDesc`

describes a textual witness or edition. It is historically useful, but it is **not evidence that the poem was composed in that year**.

We therefore classify TEI dates conservatively rather than assigning them to `publication_year`.


In [ ]:
if len(tei_dates):
    def classify_tei_date(row):
        ctx = str(row["context"]).lower()
        if "witness" in ctx or "listwit" in ctx:
            return "witness_or_edition_date"
        if "sourcedesc" in ctx or "bibl" in ctx:
            return "bibliographic_date"
        if any(k in ctx for k in ["creation","origdate","profiledesc"]):
            return "possible_work_date_needs_review"
        return "unclassified_needs_review"

    tei_dates["date_role"] = tei_dates.apply(classify_tei_date, axis=1)
    display(
        tei_dates.groupby("date_role")
        .agg(records=("file","size"), files=("file","nunique"))
        .reset_index()
        .sort_values("files", ascending=False)
    )
else:
    print("Nothing to classify.")


## 06. Corpus reconciliation checkpoint

We now compare the two corpora at the author level and explicitly test the key names for this paper.

This is not yet text matching. The next implementation step will match poems using normalized text signatures for the overlap and preserve unmatched Hernández poems (especially Herrera/Pacheco) as legitimate corpus records.


In [ ]:
h_author_labels = sorted(h_author_counts["author_file"].astype(str).unique())
n_author_labels = sorted(n_poems["author_dir"].astype(str).unique())

def contains_author(labels, key):
    nk = norm_letters(key)
    return [x for x in labels if nk in norm_letters(x)]

targets = ["Herrera","Pacheco","Garcilaso","Gongora","Quevedo","Cervantes","Lope"]
target_rows = []
for key in targets:
    target_rows.append({
        "target": key,
        "hernandez_matches": ", ".join(contains_author(h_author_labels, key)),
        "navarro_matches": ", ".join(contains_author(n_author_labels, key)),
    })
display(pd.DataFrame(target_rows))

print(f"Hernández author-file labels: {len(h_author_labels)}")
print(f"Navarro author folders: {len(n_author_labels)}")


## 07. Decision rule for the temporal design

The project will use a **hierarchy of temporal evidence**, not a single crude date.

For each poem we ultimately want:

`date_min, date_max, date_type, temporal_confidence, date_source`

with the following evidence hierarchy:

1. **Composition date / bounded composition interval** from a scholarly critical source — strongest.
2. **First publication / collection date** when it genuinely dates circulation of the poem — useful, but explicitly labelled as publication/circulation.
3. **Witness or edition date** — bibliographic evidence only; never silently converted into composition date.
4. **Author career interval** — fallback uncertainty interval, not a point estimate.
5. **Birth year** — biographical covariate only, not poem chronology.

The dynamic-network analysis will be attempted only after we know how many poems occupy levels 1–2 and how wide level-4 intervals are.


In [ ]:
summary = pd.DataFrame([
    {
        "source": "Hernández-Lorenzo",
        "unit": "recoverable poem blocks from aggregate TXT",
        "records": len(h_poems),
        "authors_or_files": len(h_author_labels),
        "temporal_evidence": "author lifespan metadata only at present",
        "proposed_role": "standardized textual backbone + Herrera/Pacheco"
    },
    {
        "source": "Navarro TEI",
        "unit": "individual TEI sonnets",
        "records": len(n_poems),
        "authors_or_files": len(n_author_labels),
        "temporal_evidence": (
            f"{tei_dates['file'].nunique()} files with exact TEI temporal records"
            if len(tei_dates) else "no exact TEI temporal records"
        ),
        "proposed_role": "title/source/witness enrichment + overlap validation"
    }
])
display(summary)

print("\nSCIENTIFIC CHECKPOINT")
print("---------------------")
print("1. Do not use Hernández lifespan 'Date' as poem time.")
print("2. Do not interpret witness/edition years as composition years.")
print("3. If Hernández blank-line recovery is consistent, use it as the text backbone.")
print("4. Next: poem-level reconciliation by normalized text + quantify genuinely datable material.")


## 08. Outputs to report back before the next code update

After running all cells, save this notebook to GitHub. The decisive outputs are:

- recovered Hernández poem blocks and their 14-line consistency;
- sanity-check matches against metadata totals;
- number and contexts of **exact** TEI date records;
- author availability for Herrera, Pacheco and the canonical overlap;
- final corpus summary.

From those outputs we will choose the temporal reconstruction strategy without inventing chronology.
